In [1]:
from datasets import load_from_disk
ds = load_from_disk("D:/sem 2/DL/project/ContractSense-copilot/data/raw/cuad")
sample = ds["train"][0]
print(sample.keys())
print(type(sample["pdf"]))
print(sample["pdf"])

d:\sem 2\DL\project\ContractSense-copilot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['pdf'])
<class 'pdfplumber.pdf.PDF'>


In [2]:
pdf = sample["pdf"]
print("pages:", len(pdf.pages))
print(pdf.pages[0].extract_text()[:2000])

pages: 11
Datasheet for Contract Understanding Atticus Dataset (CUAD)
I.MOTIVATION
A. Who created the dataset (e.g., which team, research group) and on behalf of which entity (e.g. company,
institution, organization)?
The Atticus Project is a non-profit organization whose mission is to harness the power of AI to accelerate
accurate and efficient contract review. The Atticus Project started as a grassroots movement by experienced
lawyers in public companies and leading law firms aiming to achieve high-quality, low-cost, accurate and timely
contract review using AI. It was officially incorporated as a California nonprofit public benefit corporation in
January 2020.
B. Did they fund it themselves? If there is an associated grant, please provide the name of the grantor and
the grant name and number.
The Atticus Project relies 100% on unpaid volunteers who are organized around the single mission of changing
the legal industry by leveraging AI.
C. For what purpose was the data set created? W

In [3]:
from pathlib import Path
import json

out_dir = Path("D:/sem 2/DL/project/ContractSense-copilot/data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

records = []

for i, item in enumerate(ds["train"]):
    pdf = item["pdf"]
    pages = []
    for p in pdf.pages:
        t = p.extract_text() or ""
        pages.append(t)
    full_text = "\n".join(pages).strip()

    records.append({
        "contract_id": f"cuad_{i:04d}",
        "num_pages": len(pdf.pages),
        "char_count": len(full_text),
        "text": full_text,
    })

with open(out_dir / "sample_contract_texts.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("saved:", out_dir / "sample_contract_texts.jsonl")
print("contracts saved:", len(records))
print("chars in first doc:", records[0]["char_count"])

saved: D:\sem 2\DL\project\ContractSense-copilot\data\processed\sample_contract_texts.jsonl
contracts saved: 511
chars in first doc: 32359


In [4]:
print(records[0]["text"][:2000])

Datasheet for Contract Understanding Atticus Dataset (CUAD)
I.MOTIVATION
A. Who created the dataset (e.g., which team, research group) and on behalf of which entity (e.g. company,
institution, organization)?
The Atticus Project is a non-profit organization whose mission is to harness the power of AI to accelerate
accurate and efficient contract review. The Atticus Project started as a grassroots movement by experienced
lawyers in public companies and leading law firms aiming to achieve high-quality, low-cost, accurate and timely
contract review using AI. It was officially incorporated as a California nonprofit public benefit corporation in
January 2020.
B. Did they fund it themselves? If there is an associated grant, please provide the name of the grantor and
the grant name and number.
The Atticus Project relies 100% on unpaid volunteers who are organized around the single mission of changing
the legal industry by leveraging AI.
C. For what purpose was the data set created? Was there a

In [8]:
from pathlib import Path
import runpy

module = runpy.run_path("D:/sem 2/DL/project/ContractSense-copilot/src/ingestion/clause_segmenter.py")
iter_cuad_clauses = module["iter_cuad_clauses"]
write_jsonl = module["write_jsonl"]

count = write_jsonl(
    iter_cuad_clauses("D:/sem 2/DL/project/ContractSense-copilot/data/raw/cuad"),
    "D:/sem 2/DL/project/ContractSense-copilot/data/processed/clauses.jsonl",
)

print("clause records written:", count)

clause records written: 23564
